<a href="https://colab.research.google.com/github/VicenteJimenezSanMartin-tech/Optimizaci-n/blob/main/Control_1_Vicente_Jimenez.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Control 1 con AMPL
Nombre: Vicente Jimenez San Martin

RUT: 21.116.770-k

NRC: 7920

Resolución del control 1 aplicando AMPL, mostrando un análisis de sensibilidad de los resultados y de las restricciones.

In [33]:
%pip install amplpy ipywidgets
from amplpy import ampl_notebook

def create_ampl_instance(solver: str = "highs"):
    #Descarga e inicializa el motor AMPL y el solver automáticamente en la nube local
    ampl = ampl_notebook(modules=[solver], license_uuid="default")
    ampl.option["solver"] = solver
    return ampl

## 1. Problema de La Química
Se busca minimizar el costo de dos procesos para cumplir demandas mínimas de tres productos químicos. Como es un modelo continuo, podemos analizar los precios sombra de las restricciones de demanda.

Proceso 1 (por hora):
Costo: 40 USD
Produce: 3A, 1B, 1C
Proceso 2 (por hora):
Costo: 10 USD
Produce: 1A, 1B, 0C

**Se deben cumplir demandas mínimas diarias:**

A ≥ 40
B ≥ 15
C ≥ 5

**Variables de decisión**

$x_1$: horas de uso del proceso 1

$x_2$: horas de uso del proceso 2

**Función objetivo**

Z = 40$x_1$ + 10$x_2$

**Restricciones**

Producción de A:
3$x_1$ + $x_2$ ≥ 40

Produccón de B:
$x_1$ + $x_2$ ≥ 15

Producción de C:
$x_1$ ≥ 5

No negatividad:
$x_1$, $x_2$ ≥ 0

In [34]:
# Crear la instancia de AMPL
ampl_quimica = create_ampl_instance()

# Inyectar el modelo en sintaxis pura de AMPL
ampl_quimica.eval(r'''
    set PROCESSES ordered;
    set PRODUCTS ordered;

    param cost {PROCESSES} >= 0;
    param demand {PRODUCTS} >= 0;
    param yield_rate {PRODUCTS, PROCESSES} >= 0;

    var Time {PROCESSES} >= 0;

    minimize TotalCost:
        sum {p in PROCESSES} cost[p] * Time[p];

    subject to MeetDemand {prod in PRODUCTS}:
        sum {p in PROCESSES} yield_rate[prod, p] * Time[p] >= demand[prod];
''')

# Pasar los datos
ampl_quimica.set["PROCESSES"] = ["Proceso_1", "Proceso_2"]
ampl_quimica.set["PRODUCTS"] = ["A", "B", "C"]
ampl_quimica.param["cost"] = {"Proceso_1": 40, "Proceso_2": 10}
ampl_quimica.param["demand"] = {"A": 40, "B": 15, "C": 5}
ampl_quimica.param["yield_rate"] = {
    ("A", "Proceso_1"): 3, ("A", "Proceso_2"): 1,
    ("B", "Proceso_1"): 1, ("B", "Proceso_2"): 1,
    ("C", "Proceso_1"): 1, ("C", "Proceso_2"): 0
}

# Resolver
ampl_quimica.solve()

# Mostrar Resultados
print("\nRESULTADOS LA QUÍMICA")
print(f"Horas Proceso 1: {ampl_quimica.get_variable('Time')['Proceso_1'].value():.2f}")
print(f"Horas Proceso 2: {ampl_quimica.get_variable('Time')['Proceso_2'].value():.2f}")
print(f"Costo Mínimo Total: {ampl_quimica.get_objective('TotalCost').value():.2f} USD")

print("\nANÁLISIS DE SENSIBILIDAD")
duales_demanda = ampl_quimica.get_constraint('MeetDemand').get_values().to_dict()
for producto, precio_sombra in duales_demanda.items():
    if precio_sombra != 0:
        print(f"Demanda de {producto}: {precio_sombra:.2f}")

Using default Community Edition License for Colab. Get yours at: https://ampl.com/ce
Licensed to AMPL Community Edition License for the AMPL Model Colaboratory (https://ampl.com/colab).
HiGHS 1.11.0: HiGHS 1.11.0: optimal solution; objective 450
0 simplex iterations
0 barrier iterations

RESULTADOS LA QUÍMICA
Horas Proceso 1: 5.00
Horas Proceso 2: 25.00
Costo Mínimo Total: 450.00 USD

ANÁLISIS DE SENSIBILIDAD
Demanda de A: 10.00
Demanda de C: 10.00


##Interpretación:

**Estrategia Óptima:** Para minimizar los costos al máximo, la empresa química debe ejecutar el Proceso 1 durante 5 horas y el Proceso 2 durante 25 horas diarias. Esta combinación garantiza el cumplimiento exacto de las cuotas sin desperdiciar recursos, logrando un costo total diario de 450 USD.

**Análisis de Sensibilidad:** > * Demanda de A (Precio Sombra = 10.00): Si el cliente exigiera 1 unidad extra del producto A (pasando de 40 a 41), el costo mínimo del plan de producción subiría en 10 USD.

Demanda de C (Precio Sombra = 10.00): Si se exigiera 1 unidad extra del producto C (pasando de 5 a 6), el costo mínimo también subiría en 10 USD.

La Demanda B no genera precio sombra (es 0) porque al ejecutar las horas óptimas, se producen naturalmente 30 unidades de B, lo cual supera holgadamente las 15 mínimas requeridas. Aumentar su exigencia no afectaría el presupuesto.

## 2. Problema de la Encuesta Telefónica
Modelo con variables enteras para determinar la cantidad de llamadas diurnas y nocturnas. Al ser programación entera pura, el análisis de sensibilidad tradicional mediante dualidad no es matemáticamente exacto.

**Costo por llamada:**

Día: 2 USD  
Noche: 5 USD

**Se deben cumplir los siguientes mínimos:**

Esposas ≥ 150  
Esposos ≥ 120  
Varones ≥ 100  
Mujeres ≥ 110

**Variables:**

$x_d$: llamadas en el día  
$x_n$: llamadas en la noche

**Función objetivo**

min Z = 2$x_d$ + 5$x_n$

**Restricciones**

0.30$x_d$ + 0.30$x_n$ ≥ 150

0.10$x_d$ + 0.30$x_n$ ≥ 120

0.10$x_d$ + 0.15$x_n$ ≥ 100

0.10$x_d$ + 0.20$x_n$ ≥ 110

$x_n$ ≤ 0.5($x_d$ + $x_n$)

**Naturaleza**

$x_d, x_n \in \mathbb Z^+$


In [35]:
# Crear la instancia de AMPL
ampl_encuesta = create_ampl_instance()

# Inyectar el modelo en sintaxis
ampl_encuesta.eval(r'''
    var DayCalls integer >= 0;
    var NightCalls integer >= 0;

    minimize TotalCost:
        2 * DayCalls + 5 * NightCalls;

    subject to MinWives:
        0.30 * DayCalls + 0.30 * NightCalls >= 150;

    subject to MinHusbands:
        0.10 * DayCalls + 0.30 * NightCalls >= 120;

    subject to MinMales:
        0.10 * DayCalls + 0.15 * NightCalls >= 100;

    subject to MinFemales:
        0.10 * DayCalls + 0.20 * NightCalls >= 110;

    subject to NightLimit:
        NightCalls <= 0.5 * (DayCalls + NightCalls);
''')

# Resolver
ampl_encuesta.solve()

# Mostrar Resultados
print("\n--- RESULTADOS ENCUESTA TELEFÓNICA ---")
print(f"Llamadas de Día (x_d): {ampl_encuesta.get_variable('DayCalls').value():.0f}")
print(f"Llamadas de Noche (x_n): {ampl_encuesta.get_variable('NightCalls').value():.0f}")
print(f"Costo Mínimo Total: {ampl_encuesta.get_objective('TotalCost').value():.2f} USD")


Using default Community Edition License for Colab. Get yours at: https://ampl.com/ce
Licensed to AMPL Community Edition License for the AMPL Model Colaboratory (https://ampl.com/colab).
HiGHS 1.11.0: HiGHS 1.11.0: optimal solution; objective 2300
3 simplex iterations
1 branching nodes

--- RESULTADOS ENCUESTA TELEFÓNICA ---
Llamadas de Día (x_d): 900
Llamadas de Noche (x_n): 100
Costo Mínimo Total: 2300.00 USD


##Interpretación:

**Estrategia Óptima:** Para minimizar los costos, la empresa encuestadora debe realizar 900 llamadas en el día y 100 llamadas en la noche. Esta combinación es altamente eficiente, ya que logra cumplir holgadamente con la cuota de esposas y varones, y calza de manera exacta con los mínimos requeridos de esposos 120 y mujeres 110. El costo mínimo garantizado para esta operación es de 2300 USD.

**Análisis de Sensibilidad:** Al ser un modelo con variables enteras puras las llamadas no se pueden fraccionar, el análisis clásico de precios sombra no se aplica. Sin embargo, el análisis del comportamiento del modelo demuestra que las llamadas nocturnas, a pesar de ser un 150% más caras que las diurnas, son indispensables para la optimización. Su alta tasa de respuesta en los segmentos de "esposos" y "mujeres" permite reducir el volumen total de llamadas, ahorrando presupuesto en comparación a una estrategia exclusivamente diurna.


## 3. Problema de Reciclaje Industrial

Se busca minimizar el costo de mezcla de las dos chatarras para cumplir con proporciones químicas exactas, utilizando parámetros y conjuntos ordenados estilo matriz.

**Variables de Decisión**

$x_A$: Toneladas de chatarra A a utilizar.

$x_B$: Toneladas de chatarra B a utilizar.

**Función Objetivo**

Minimizar el costo total de las chatarras:

min Z = 100$x_A$ + 80$x_B$

**Restricciones**

Peso Total:
La suma de las chatarras debe ser exactamente 1000 toneladas.
$x_A$ + $x_B$ = 1000

**Límites de Aluminio:**
0.03 ≤ $\frac{0.06x_A + 0.03x_B}{1000}$ ≤ 0.06

**Límites de Silicio:**
0.03 ≤ $\frac{0.03x_A + 0.06x_B}{1000}$ ≤ 0.05

**Límites de Carbón:**
0.03 ≤ $\frac{0.04x_A + 0.03x_B}{1000}$ ≤ 0.07

**No negatividad:**
$x_A$ ≥ 0, $x_B$ ≥ 0

In [36]:
ampl_reciclaje = create_ampl_instance()
ampl_reciclaje.eval(r'''
    set SCRAPS ordered;
    param cost {SCRAPS} >= 0;
    param al {SCRAPS} >= 0;
    param si {SCRAPS} >= 0;
    param ca {SCRAPS} >= 0;

    var Use {SCRAPS} >= 0;

    minimize TotalCost:
        sum {s in SCRAPS} cost[s] * Use[s];

    subject to TotalWeight:
        sum {s in SCRAPS} Use[s] = 1000;

    subject to MinAluminum:
        sum {s in SCRAPS} al[s] * Use[s] >= 0.03 * 1000;

    subject to MaxAluminum:
        sum {s in SCRAPS} al[s] * Use[s] <= 0.06 * 1000;

    subject to MinSilicon:
        sum {s in SCRAPS} si[s] * Use[s] >= 0.03 * 1000;

    subject to MaxSilicon:
        sum {s in SCRAPS} si[s] * Use[s] <= 0.05 * 1000;

    subject to MinCarbon:
        sum {s in SCRAPS} ca[s] * Use[s] >= 0.03 * 1000;

    subject to MaxCarbon:
        sum {s in SCRAPS} ca[s] * Use[s] <= 0.07 * 1000;
''')

ampl_reciclaje.set["SCRAPS"] = ["A", "B"]
ampl_reciclaje.param["cost"] = {"A": 100, "B": 80}
ampl_reciclaje.param["al"] = {"A": 0.06, "B": 0.03}
ampl_reciclaje.param["si"] = {"A": 0.03, "B": 0.06}
ampl_reciclaje.param["ca"] = {"A": 0.04, "B": 0.03}

ampl_reciclaje.solve()

print("\nRESULTADOS ALEACIÓN")
for s in ["A", "B"]:
    print(f"Chatarra {s}: {ampl_reciclaje.get_variable('Use')[s].value():.2f} toneladas")
print(f"Costo Mínimo: {ampl_reciclaje.get_objective('TotalCost').value():.2f} USD")

print("\nANÁLISIS DE SENSIBILIDAD")
for name, constraint in ampl_reciclaje.get_constraints():
    if constraint.dual() != 0:
        print(f"{name}: {constraint.dual():.2f}")

Using default Community Edition License for Colab. Get yours at: https://ampl.com/ce
Licensed to AMPL Community Edition License for the AMPL Model Colaboratory (https://ampl.com/colab).
HiGHS 1.11.0: HiGHS 1.11.0: optimal solution; objective 86666.66667
0 simplex iterations
0 barrier iterations

RESULTADOS ALEACIÓN
Chatarra A: 333.33 toneladas
Chatarra B: 666.67 toneladas
Costo Mínimo: 86666.67 USD

ANÁLISIS DE SENSIBILIDAD
TotalWeight: 120.00
MaxSilicon: -666.67


##Interpretación:

**Estrategia Óptima:** El modelo determina que se deben mezclar 333.33 toneladas de la Chatarra A con 666.67 toneladas de la Chatarra B. El sistema prioriza el uso de la Chatarra B porque es más económica 80 USD, llevándola al límite máximo permitido antes de romper las reglas químicas. El costo mínimo es de 86,666.67 USD.

**Análisis de Sensibilidad:**

**TotalWeight 120.00:** Si se necesitara producir 1 tonelada extra de aleación 1001 toneladas totales, el costo aumentaría en 120 USD.

**MaxSilicon -666.67:** El tope de silicio es la restricción que nos impide usar más chatarra económica B. El signo negativo indica un ahorro: si las normativas permitieran un límite de silicio ligeramente superior en solo 1 unidad, el costo de producción se reduciría a una tasa de 666.67 USD al poder usar más chatarra B y menos de la A.

## 4. Problema Donovan Licuadoras

Se busca minimizar los costos de mano de obra e inventario garantizando el cumplimiento de la demanda trimestral.

**Variables de Decisión**

$E$: Número total de empleados contratados al año.

$w_t$: Número de empleados trabajando activamente en el trimestre $t \in \{1,2,3,4\}$.

$v_t$: Número de empleados de vacaciones en el trimestre $t \in \{1,2,3,4\}$.

$P_t$: Cantidad de licuadoras producidas en el trimestre $t \in \{1,2,3,4\}$.

$I_t$: Inventario de licuadoras al final del trimestre $t \in \{0,1,2,3,4\}$.

**Función Objetivo**

Minimizar el costo total anual (salarios de todos los contratados + almacenamiento en inventario):

min Z = 30000$E$ + 30 $\sum_{t=1}^{4} I_t$

**Restricciones**

**Equilibrio de Trabajadores:**
En cada trimestre, un empleado contratado o está trabajando o está de vacaciones.

$w_t$ + $v_t$ = $E$ $\quad \forall t$ $\in \{1, 2, 3, 4\}$

**Política de Vacaciones:**
A lo largo del año, el total de trimestres de vacaciones otorgados debe ser igual al total de empleados (cada persona toma exactamente un trimestre libre).

$\sum_{t=1}^{4}$ $v_t$ = $E$

**Límite de Producción:**
La producción de un trimestre está limitada por los empleados que están trabajando activamente (máximo 500 licuadoras por persona).

$P_t$ ≤ 500$w_t$ $\quad \forall t$ $\in \{1, 2, 3, 4\}$

**Conservación (Flujo) de Inventario:**
El inventario anterior más la producción actual debe igualar a la demanda de ese trimestre más el inventario que sobra.

$I_{t-1}$ + $P_t$ = $D_t$ + $I_t$ $\quad \forall t$ $\in \{1, 2, 3, 4\}$

(Donde las demandas son $D_1$=4000, $D_2$=2000, $D_3$=3000, $D_4$=10000)

**Inventario Inicial:**
Hay en existencia 600 licuadoras al comenzar el año.

$I_0$ = 600

**Naturaleza de las Variables (Dominio):**
Los empleados (activos, totales y en vacaciones) y la producción deben ser números enteros positivos.

$E$, $w_t$, $v_t$, $P_t$ $\in \mathbb{Z}^+$

$I_t$ ≥ 0

In [37]:
ampl_donovan = create_ampl_instance()
ampl_donovan.eval(r'''
    set QUARTERS ordered;
    param demand {QUARTERS} >= 0;
    param init_inv >= 0;

    var EmpTotal integer >= 0;
    var Vacations {QUARTERS} integer >= 0;
    var Work {QUARTERS} integer >= 0;
    var Produce {QUARTERS} integer >= 0;
    var Inv {0..4} >= 0;

    minimize TotalCost:
        30000 * EmpTotal + sum {q in QUARTERS} 30 * Inv[q];

    subject to EmpBalance {q in QUARTERS}:
        Work[q] + Vacations[q] = EmpTotal;

    subject to VacationLimit:
        sum {q in QUARTERS} Vacations[q] = EmpTotal;

    subject to ProductionLimit {q in QUARTERS}:
        Produce[q] <= 500 * Work[q];

    subject to InitInventory:
        Inv[0] = init_inv;

    subject to InventoryFlow {q in QUARTERS}:
        Inv[q-1] + Produce[q] = demand[q] + Inv[q];
''')

ampl_donovan.set["QUARTERS"] = [1, 2, 3, 4]
ampl_donovan.param["demand"] = {1: 4000, 2: 2000, 3: 3000, 4: 10000}
ampl_donovan.param["init_inv"] = 600

ampl_donovan.solve()

print("\nRESULTADOS DONOVAN")
print(f"Empleados Totales Contratados: {ampl_donovan.get_variable('EmpTotal').value()}")
print(f"Costo Anual: {ampl_donovan.get_objective('TotalCost').value()} USD")

Using default Community Edition License for Colab. Get yours at: https://ampl.com/ce
Licensed to AMPL Community Edition License for the AMPL Model Colaboratory (https://ampl.com/colab).
HiGHS 1.11.0: HiGHS 1.11.0: optimal solution; objective 495000
7 simplex iterations
1 branching nodes

RESULTADOS DONOVAN
Empleados Totales Contratados: 13.0
Costo Anual: 495000.0 USD


##Interpretación:

**Estrategia Óptima:** La empresa debe contratar exactamente a 13 empleados para todo el año, lo que genera un costo fijo en salarios de 390,000 USD. Dado que en el trimestre 4 la demanda de 10,000 licuadoras supera la capacidad productiva de los empleados activos, la empresa está obligada a producir por adelantado y almacenar un inventario de 3,500 licuadoras al final del trimestre 3. El costo de guardar este inventario es de 105,000 USD, lo que suma el costo total óptimo de 495,000 USD.

## 5. Problema Refinería Melrose

Un problema clásico de "blending" (mezcla). Transformamos las restricciones de "grado promedio" multiplicando los denominadores para mantener la linealidad pura del modelo.

**Variables de Decisión**

$P_m$: Barriles de crudo procesados por el método $m \in \{1, 2, 3\}$.

$F_{6 \to 8}$: Barriles de grado 6 fraccionados para convertirse en grado 8.

$F_{8 \to 10}$: Barriles de grado 8 fraccionados para convertirse en grado 10.

$G_g$: Barriles de grado $g \in \{6, 8, 10\}$ destinados a gasolina.

$A_g$: Barriles de grado $g \in \{6, 8, 10\}$ destinados a aceite combustible.

$W_g$: Barriles de grado $g \in \{6, 8, 10\}$ desechados.

**Función Objetivo**

Maximizar la utilidad total (ingresos por ventas - costos de proceso - costos de fraccionamiento - costos de desecho):

max Z = 8 $\sum_{g} G_g$ + 6 $\sum_{g} A_g$ - $\sum_{m} C_m P_m$ - 1.30 $F_{6 \to 8}$ - 2.00 $F_{8 \to 10}$ - 0.20 $\sum_{g} W_g$

(Donde los costos de proceso son $C_1$=3.4, $C_2$=3.0, $C_3$=2.6)

**Restricciones**

**Balance Grado 6:**
Todo lo que se produce de grado 6 por los tres métodos debe ser igual a lo que se destina a gasolina, aceite, se fracciona a grado 8 o se desecha.

0.2$P_1$ + 0.3$P_2$ + 0.4$P_3$ = $F_{6 \to 8}$ + $G_6$ + $A_6$ + $W_6$

**Balance Grado 8:**
La producción de grado 8 más lo que entra convertido desde el grado 6 debe igualar a lo destinado a gasolina, aceite, convertido a 10 o desechado.

0.2$P_1$ + 0.3$P_2$ + 0.4$P_3$ + $F_{6 \to 8}$ = $F_{8 \to 10}$ + $G_8$ + $A_8$ + $W_8$

**Balance Grado 10:**
La producción de grado 10 más lo que entra convertido desde el grado 8 debe igualar a lo que se destina a gasolina, aceite o desecho.

0.6$P_1$ + 0.4$P_2$ + 0.2$P_3$ + $F_{8 \to 10}$ = $G_{10}$ + $A_{10}$ + $W_{10}$

**Calidad de Gasolina:**
El grado promedio de la gasolina producida debe ser por lo menos 9.

6$G_6$ + 8$G_8$ + 10$G_{10}$ ≥ 9($G_6$ + $G_8$ + $G_{10}$)

**Calidad de Aceite Combustible:**
El grado promedio del aceite producido debe ser por lo menos 7.

6$A_6$ + 8$A_8$ + 10$A_{10}$ ≥ 7($A_6$ + $A_8$ + $A_{10}$)

**Límites de Mercado:**
Se pueden vender como máximo 2000 barriles de gasolina y 600 de aceite combustible.

$\sum_{g} G_g$ ≤ 2000

$\sum_{g} A_g$ ≤ 600

**Naturaleza de las Variables (Dominio):**
Todas las variables deben ser números continuos positivos o cero.

$P_m$, $F_{6 \to 8}$, $F_{8 \to 10}$, $G_g$, $A_g$, $W_g$ ≥ 0

In [38]:
ampl_melrose = create_ampl_instance()
ampl_melrose.eval(r'''
    set METHODS ordered;
    set GRADES ordered;

    param cost_process {METHODS} >= 0;
    param yield_grade {METHODS, GRADES} >= 0;

    var Process {METHODS} >= 0;
    var Frac6to8 >= 0;
    var Frac8to10 >= 0;
    var SellGas {GRADES} >= 0;
    var SellFuel {GRADES} >= 0;
    var Waste {GRADES} >= 0;

    maximize TotalProfit:
        8 * (sum {g in GRADES} SellGas[g]) + 6 * (sum {g in GRADES} SellFuel[g])
        - sum {m in METHODS} cost_process[m] * Process[m]
        - 1.30 * Frac6to8 - 2.00 * Frac8to10
        - 0.20 * sum {g in GRADES} Waste[g];

    subject to Balance6:
        sum {m in METHODS} yield_grade[m, 6] * Process[m] = Frac6to8 + SellGas[6] + SellFuel[6] + Waste[6];

    subject to Balance8:
        sum {m in METHODS} yield_grade[m, 8] * Process[m] + Frac6to8 = Frac8to10 + SellGas[8] + SellFuel[8] + Waste[8];

    subject to Balance10:
        sum {m in METHODS} yield_grade[m, 10] * Process[m] + Frac8to10 = SellGas[10] + SellFuel[10] + Waste[10];

    subject to GasQuality:
        6 * SellGas[6] + 8 * SellGas[8] + 10 * SellGas[10] >= 9 * sum {g in GRADES} SellGas[g];

    subject to FuelQuality:
        6 * SellFuel[6] + 8 * SellFuel[8] + 10 * SellFuel[10] >= 7 * sum {g in GRADES} SellFuel[g];

    subject to MaxGas:
        sum {g in GRADES} SellGas[g] <= 2000;

    subject to MaxFuel:
        sum {g in GRADES} SellFuel[g] <= 600;
''')

ampl_melrose.set["METHODS"] = [1, 2, 3]
ampl_melrose.set["GRADES"] = [6, 8, 10]
ampl_melrose.param["cost_process"] = {1: 3.4, 2: 3.0, 3: 2.6}
ampl_melrose.param["yield_grade"] = {
    (1, 6): 0.2, (1, 8): 0.2, (1, 10): 0.6,
    (2, 6): 0.3, (2, 8): 0.3, (2, 10): 0.4,
    (3, 6): 0.4, (3, 8): 0.4, (3, 10): 0.2
}

ampl_melrose.solve()

print("\nRESULTADOS MELROSE")
print(f"Utilidad Máxima: {ampl_melrose.get_objective('TotalProfit').value():.2f} USD")

print("\nANÁLISIS DE SENSIBILIDAD")
for name, constraint in ampl_melrose.get_constraints():
    if constraint.dual() != 0:
        print(f"{name}: {constraint.dual():.2f}")

Using default Community Edition License for Colab. Get yours at: https://ampl.com/ce
Licensed to AMPL Community Edition License for the AMPL Model Colaboratory (https://ampl.com/colab).
HiGHS 1.11.0: HiGHS 1.11.0: optimal solution; objective 11230
9 simplex iterations
0 barrier iterations

RESULTADOS MELROSE
Utilidad Máxima: 11230.00 USD

ANÁLISIS DE SENSIBILIDAD
Balance6: -1.55
Balance8: -2.85
Balance10: -4.20
GasQuality: -0.67
FuelQuality: -0.65
MaxGas: 4.48
MaxFuel: 3.80


##Interpretación:

**Estrategia Óptima:** Evaluando las combinaciones de fraccionamiento y los costos por desecho, la refinería alcanza una utilidad neta máxima de 11,230.00 USD.

**Análisis de Sensibilidad:**

**MaxGas 4.48 y MaxFuel 3.80:** Estas son las ganancias marginales. Si el mercado permitiera vender un barril extra de gasolina superando los 2000, la utilidad neta subiría en 4.48 USD. Si se pudiera vender un barril extra de aceite combustible, subiría 3.80 USD.

**GasQuality -0.67 y FuelQuality -0.65:** Los precios sombra negativos en maximización representan una penalización. La exigencia estricta de mantener el grado promedio de la gasolina por encima de 9 nos "cuesta" 0.67 USD por unidad restrictiva. Si la norma fuera más flexible, la ganancia aumentaría.

## 6. Problema Maderera Brady

Optimización de la cadena de suministro maderera, comparando compra externa vs tala propia, sujeto a capacidades físicas de procesamiento y secado. Es clave la conversión de las 40 horas disponibles del horno a segundos para mantener la consistencia de las unidades.

**Variables de Decisión**

$G_1$: Pies cúbicos de tablones de Grado 1 comprados a la semana.

$G_2$: Pies cúbicos de tablones de Grado 2 comprados a la semana.

$T$: Pies cúbicos de troncos propios talados y procesados a la semana.

**Función Objetivo**

Minimizar el costo total semanal, que incluye los costos de compra ($3 para G1, $7 para G2), el costo de talar troncos ($3), el costo del aserradero para troncos ($2.50), y el costo de secado para toda la madera ($4):

min Z = 3$G_1$ + 7$G_2$ + 3$T$ + 2.5$T$ + 4($G_1$ + $G_2$ + $T$)

*(Simplificado analíticamente como: 7$G_1$ + 11$G_2$ + 9.5$T$)*

**Restricciones**

**Demanda de Madera Útil:**
Los tablones rinden diferentes porcentajes de madera útil luego del secado. En total, se deben obtener al menos 90,000 pies cúbicos de madera útil.

0.7$G_1$ + 0.9$G_2$ + 0.8$T$ ≥ 90000

**Límites de Mercado (Proveedores):**
Se pueden comprar como máximo 40,000 pies cúbicos de Grado 1 y 60,000 de Grado 2 a la semana.

$G_1$ ≤ 40000

$G_2$ ≤ 60000

**Capacidad del Aserradero Propio:**
Solo aplica a los troncos propios talados.

$T$ ≤ 35000

**Capacidad de Tiempo del Horno (Equivalencia en segundos):**
Se dispone de 40 horas a la semana. Dado que los tiempos de secado están en segundos, igualamos las unidades (40 horas $\times$ 60 minutos $\times$ 60 segundos = 144,000 segundos).

1.2$G_1$ + 0.8$G_2$ + 1.3$T$ ≤ 144000

**Naturaleza de las Variables (Dominio):**
Las variables representan dimensiones físicas continuas.

$G_1$, $G_2$, $T$ ≥ 0

In [39]:
ampl_brady = create_ampl_instance()
ampl_brady.eval(r'''
    var Grade1 >= 0;
    var Grade2 >= 0;
    var Logs >= 0;

    minimize TotalCost:
        3 * Grade1 + 7 * Grade2 + 3 * Logs + 4 * (Grade1 + Grade2 + Logs) + 2.5 * Logs;

    subject to Demand:
        0.7 * Grade1 + 0.9 * Grade2 + 0.8 * Logs >= 90000;

    subject to MaxGrade1:
        Grade1 <= 40000;

    subject to MaxGrade2:
        Grade2 <= 60000;

    subject to MaxLogs:
        Logs <= 35000;

    subject to DryingTime:
        1.2 * Grade1 + 0.8 * Grade2 + 1.3 * Logs <= 144000;
''')

ampl_brady.solve()

print("\nRESULTADOS BRADY")
print(f"Comprar Grado 1: {ampl_brady.get_variable('Grade1').value():.2f} pies cubicos")
print(f"Comprar Grado 2: {ampl_brady.get_variable('Grade2').value():.2f} pies cubicos")
print(f"Procesar Troncos: {ampl_brady.get_variable('Logs').value():.2f} pies cubicos")
print(f"Costo Total: {ampl_brady.get_objective('TotalCost').value():.2f} USD")

print("\nANÁLISIS DE SENSIBILIDAD")
for name, constraint in ampl_brady.get_constraints():
    print(f"{name}: {constraint.dual():.2f}")

Using default Community Edition License for Colab. Get yours at: https://ampl.com/ce
Licensed to AMPL Community Edition License for the AMPL Model Colaboratory (https://ampl.com/colab).
HiGHS 1.11.0: HiGHS 1.11.0: optimal solution; objective 1028055.556
0 simplex iterations
0 barrier iterations

RESULTADOS BRADY
Comprar Grado 1: 40000.00 pies cubicos
Comprar Grado 2: 37777.78 pies cubicos
Procesar Troncos: 35000.00 pies cubicos
Costo Total: 1028055.56 USD

ANÁLISIS DE SENSIBILIDAD
Demand: 12.22
MaxGrade1: -1.56
MaxGrade2: 0.00
MaxLogs: -0.28
DryingTime: 0.00


##Interpretación:

**Estrategia Óptima:** La cadena de suministro se optimiza con un gasto semanal de 1,028,055.56 USD. Físicamente, la estrategia dicta explotar al máximo los recursos más eficientes: procesar toda la capacidad de Troncos propios 35,000 y comprar todo el límite disponible del Grado 1 40,000. El faltante para cumplir la meta de madera útil se cubre comprando 37,777.78 de Grado 2, sin llegar a agotar el stock del proveedor.

**Análisis de Sensibilidad:**

**Demand 12.22:** Si se exigiera 1 unidad extra de madera útil, el costo total subiría en 12.22 USD que equivale a comprar y procesar obligatoriamente la madera más cara, el Grado 2, ya que el resto está al tope.

**MaxGrade1 -1.56 y MaxLogs -0.28:** Son ahorros marginales. Si se aumentaran las capacidades máximas de estos recursos baratos, el costo se reduciría en 1.56 USD y 0.28 USD respectivamente, al sustituir madera cara por estas opciones.